# Telugu Data Cleaning Pipeline

## Strategy

**Step A: Preprocess manual data** (repair-focused, source-specific fixes)  
**Step B: Light preprocessing on downloaded data** (already formatted, just normalize)  
**Step C: Combined filtering** (language filter + heuristics + deduplication)  
**Step D: Train/Val/Test split** (90/5/5)  

Output: `../datasets/cleaned/telugu_cleaned_corpus.jsonl` — each doc labeled `type: manual/downloaded`

## 1. Setup

In [1]:
import json, re, unicodedata, hashlib, time
from pathlib import Path
from collections import Counter
from datasketch import MinHash, MinHashLSH

BASE_DIR    = Path("..").resolve()
RAW_DIR     = BASE_DIR / "datasets" / "raw"
CLEANED_DIR = BASE_DIR / "datasets" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = CLEANED_DIR / "telugu_cleaned_corpus.jsonl"

TELUGU_RE    = re.compile(r"[\u0C00-\u0C7F]")
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")

# Telugu stopwords (common particles/postpositions excluded from repeat check)
TE_STOPWORDS = {
    'ని', 'లో', 'ను', 'కు', 'కి', 'తో', 'నుండి', 'పై', 'లో', 'అని', 'మరియు',
    'ఆ', 'ఈ', 'అ', 'ఇ', 'అది', 'ఇది', 'వారు', 'మేము', 'నేను', 'అతను',
    'ఆమె', 'కానీ', 'మరి', 'అయినా', 'చేసే', 'చేశారు', 'ఉంది', 'ఉన్నారు',
    'అయింది', 'అయ్యింది', 'చేశాడు', 'చేసింది', 'వచ్చారు', 'వెళ్ళారు',
}

manual_files   = sorted(f for f in (RAW_DIR / "manual").glob("*.jsonl") if f.stat().st_size > 0)
downloaded_files = sorted(f for f in (RAW_DIR / "downloaded").glob("*.jsonl") if f.stat().st_size > 0)
print(f"Manual:     {len(manual_files)} files")
print(f"Downloaded: {len(downloaded_files)} files")


Manual:     13 files
Downloaded: 4 files


## 2. Step A: Manual Data Preprocessing

Our manual sources have specific noise:
- **Wikipedia/Wikibooks/Wiktionary**: residual `{{markup}}`, `[[links]]`, `==headers==`, `[1]` citations
- **IA books/magazines (OCR)**: column breaks, page headers/footers, garbled glyphs, ad fragments
- **Newspapers (OCR)**: column bleed, classified ads, date headers
- **All**: HTML entities, stray Devanagari (from Hindi ads), URLs

Strategy: **repair** the text rather than discarding whole documents.

In [2]:
# Patterns to clean from manual data
WIKI_MARKUP     = re.compile(r"\{\{[^}]*\}\}|\[\[[^\]|]*\|([^\]]*)\]\]|\[\[[^\]]*\]\]|<[^>]*>|==+[^=]+=+=*")
CITATION_MARKERS = re.compile(r"\[\d+\]")
PAGE_MARKERS    = re.compile(r"---\s*Page\s*\d+\s*---|\f|={5,}")
NAV_PATTERNS    = re.compile(r"[←→←→]|\bprev\b|\bnext\b|\bhome\b", re.IGNORECASE)
URL_RE          = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE        = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE        = re.compile(r"(?:\+91|91|0)?[\s.-]?[6-9]\d{9}")
MULTI_SPACE     = re.compile(r"[ \t]{2,}")

def preprocess_manual(text):
    """Repair manual data: fix OCR issues, remove markup, clean noise."""
    # Handle escape characters
    text = text.replace("\\n", "\n").replace("\\r", "\r").replace("\\t", "\t")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # HTML entities
    text = text.replace("&amp;", "&").replace("&nbsp;", " ")
    text = text.replace("&lt;", "<").replace("&gt;", ">")
    text = text.replace("&quot;", '"').replace("&#39;", "'")

    # Unicode NFC
    text = unicodedata.normalize("NFC", text)

    # Remove wiki markup (keep link text where possible)
    text = WIKI_MARKUP.sub(lambda m: m.group(1) if m.lastindex and m.group(1) else "", text)
    text = CITATION_MARKERS.sub("", text)
    text = PAGE_MARKERS.sub("", text)
    text = URL_RE.sub("", text)
    text = EMAIL_RE.sub("", text)
    text = PHONE_RE.sub("", text)

    # Remove lines with no Telugu content (navigation, English headers, stray Devanagari)
    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            cleaned_lines.append("")
            continue
        chars = line.replace(" ", "")
        if len(chars) > 5:
            telugu_count   = len(TELUGU_RE.findall(line))
            devanagari_count = len(DEVANAGARI_RE.findall(line))
            total_script = telugu_count + devanagari_count
            if total_script == 0:
                continue  # pure English/nav line — drop
            # Only strip Devanagari chars if they're clearly contaminant (>5% of line chars)
            if devanagari_count > 0 and devanagari_count / max(len(chars), 1) > 0.05:
                line = DEVANAGARI_RE.sub("", line).strip()
            cleaned_lines.append(line)
        else:
            cleaned_lines.append(line)  # keep short lines (verse numbers, dates etc)

    text = "\n".join(cleaned_lines)

    # Fix OCR column breaks (hyphenated line endings)
    text = re.sub(r"-\n", "", text)

    # Collapse whitespace
    text = MULTI_SPACE.sub(" ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# Test on one doc
with open(manual_files[0]) as f:
    sample  = json.loads(f.readline()).get("text", "")
    cleaned = preprocess_manual(sample)
    print(f"Sample: {len(sample)} → {len(cleaned)} chars")
    print(f"Preview: {cleaned[:200]}")
print("\nManual preprocessing defined.")


Sample: 101 → 101 chars
Preview: వ3కాడ౦0 1౬
[షవూగవ౫ఐ 6 (1125ల
గెటివాత[)౫ఎ 6౧౫౧1 ట

|

00-0. ౮౭00400426 142 0016060౧. 091266 0 66209011

Manual preprocessing defined.


## 3. Step B: Downloaded Data Preprocessing

Downloaded data (IndicCorp, CulturaX) is already formatted text — just needs light normalization.

In [3]:
def preprocess_downloaded(text):
    """Light normalization for already-formatted downloaded data."""
    text = text.replace("\\n", "\n").replace("\\r", "\r").replace("\\t", "\t")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = unicodedata.normalize("NFC", text)
    text = URL_RE.sub("", text)
    text = MULTI_SPACE.sub(" ", text)
    return text.strip()

print("Downloaded preprocessing defined.")


Downloaded preprocessing defined.


## 4. Step C: Combined Filtering (Language + Heuristics + Dedup)

Applied to ALL data after preprocessing.

**Changes from original:**
- `MIN_WORDS` raised to 8 (was 5) — 5-word docs add noise
- `MINHASH_MIN_WORDS` lowered to 50 (was 80) — catch more near-dups
- Word repeat check excludes Telugu stopwords (particles/postpositions)
- Short-line check: uses script-char presence not word-count length
- Exact MD5 hash before MinHash for fast exact-dup detection

In [4]:
MIN_WORDS        = 8    # was 5; 5-word docs are mostly garbage/noise
MINHASH_MIN_WORDS = 50  # was 80; catch more near-dups

def passes_language(text):
    """Keep docs with >=40% Telugu characters."""
    chars = text.replace(" ", "").replace("\n", "")
    if len(chars) < 20:
        return False
    return len(TELUGU_RE.findall(text)) / len(chars) >= 0.40

def passes_heuristics(text):
    """Quality checks with stopword-aware word repeat and script-char short-line check."""
    words = text.split()
    n     = len(words)

    if n < MIN_WORDS:
        return False, "too_short"
    if n > 100_000:
        return False, "too_long"

    # Stopword-aware word repeat (only for docs ≥50 words)
    if n >= 50:
        content_words = [w for w in words if w not in TE_STOPWORDS]
        if content_words:
            wc = Counter(content_words)
            if wc.most_common(1)[0][1] / len(content_words) > 0.45:
                return False, "word_repeat"

    # Script-char short-line check (replaces crude word-length check)
    lines = [l for l in text.split("\n") if l.strip()]
    if len(lines) > 5:
        dead = sum(1 for l in lines if len(TELUGU_RE.findall(l)) == 0 and len(l.split()) > 2)
        if dead / len(lines) > 0.75:
            return False, "menu_like"

    return True, "ok"

# Dedup: exact-match for all + MinHash for longer docs
exact_hashes = set()
lsh          = MinHashLSH(threshold=0.85, num_perm=64)
dc           = [0]

def is_duplicate(text):
    """Exact MD5 (all docs) + MinHash (50+ word docs only)."""
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    if h in exact_hashes:
        return True, "exact_dup"
    exact_hashes.add(h)
    words = text.split()
    if len(words) >= MINHASH_MIN_WORDS:
        m = MinHash(num_perm=64)
        for i in range(len(words) - 2):
            m.update(f"{words[i]} {words[i+1]} {words[i+2]}".encode("utf-8"))
        if lsh.query(m):
            return True, "near_dup"
        try:
            lsh.insert(f"d_{dc[0]}", m)
            dc[0] += 1
        except Exception:
            pass
    return False, ""

print(f"Filters + dedup defined (MIN_WORDS={MIN_WORDS}, MINHASH_MIN_WORDS={MINHASH_MIN_WORDS}, 64 perms).")


Filters + dedup defined (MIN_WORDS=8, MINHASH_MIN_WORDS=50, 64 perms).


## 5. Run Full Pipeline

In [5]:
stats = Counter()

def process_file(filepath, out_fh, dtype):
    is_manual = (dtype == "manual")
    n_docs = n_words = 0
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            stats["total"] += 1
            try:
                data = json.loads(line)
                text = data.get("text", "")
                src  = data.get("source", filepath.stem)
            except Exception:
                continue

            # Step A/B: Preprocess
            if is_manual:
                text = preprocess_manual(text)
            else:
                text = preprocess_downloaded(text)

            if not text:
                stats["drop_empty"] += 1
                continue

            # Language filter
            if not passes_language(text):
                stats["drop_language"] += 1
                continue

            # Heuristics
            ok, reason = passes_heuristics(text)
            if not ok:
                stats[f"drop_{reason}"] += 1
                continue

            # Dedup
            is_dup, dup_reason = is_duplicate(text)
            if is_dup:
                stats[f"drop_{dup_reason}"] += 1
                continue

            # Write
            w = len(text.split())
            out_fh.write(json.dumps({"text": text, "source": src, "type": dtype}, ensure_ascii=False) + "\n")
            stats[f"{dtype}_docs"]  += 1
            stats[f"{dtype}_words"] += w
            n_docs  += 1
            n_words += w

            total_ret = stats["manual_docs"] + stats["downloaded_docs"]
            if total_ret % 1_000_000 == 0 and total_ret > 0:
                print(f"  {total_ret/1e6:.1f}M retained...")

    return n_docs, n_words

start = time.time()
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    print("=== MANUAL (preprocessed + filtered) ===")
    for fp in manual_files:
        n_docs, n_words = process_file(fp, out, "manual")
        print(f"  {fp.name:<55} {n_docs:>8,} docs  {n_words/1e6:>6.2f}M words")
    print(f"  TOTAL manual: {stats['manual_docs']:,} docs, {stats['manual_words']/1e6:.1f}M words\n")

    if downloaded_files:
        print("=== DOWNLOADED (light preprocess + filtered) ===")
        for fp in downloaded_files:
            n_docs, n_words = process_file(fp, out, "downloaded")
            print(f"  {fp.name:<55} {n_docs:>8,} docs  {n_words/1e6:>6.2f}M words")
        print(f"  TOTAL downloaded: {stats['downloaded_docs']:,} docs, {stats['downloaded_words']/1e6:.1f}M words")

print(f"\nDone in {(time.time()-start)/60:.1f} min.")


=== MANUAL (preprocessed + filtered) ===
  bharatiya_chitrakala_ocr.jsonl                               227 docs    0.02M words
  navarasa_kadambari_ocr.jsonl                                  24 docs    0.00M words
  telugu_ia_books.jsonl                                    275,457 docs   21.22M words
  telugu_news_andhrajyothy.jsonl                             5,525 docs    1.54M words
  telugu_newspapers.jsonl                                   80,820 docs    3.11M words
  telugu_puranas_classical.jsonl                             2,202 docs    0.11M words
  telugu_vignana_sarvasvam.jsonl                             3,098 docs    0.07M words
  telugu_vipula_magazine.jsonl                             281,921 docs    7.08M words
  telugu_wikibooks.jsonl                                     1,314 docs    2.14M words
  telugu_wikipedia.jsonl                                    86,952 docs   34.44M words
  telugu_wikiquote.jsonl                                     1,487 docs    0.20M words
  

## 6. Summary

In [6]:
tw = stats["manual_words"] + stats["downloaded_words"]
td = stats["manual_docs"]  + stats["downloaded_docs"]
print("=" * 60)
print(f"Input:       {stats['total']:,} docs")
print(f"Output:      {td:,} docs, {tw:,} words ({tw/1e6:.1f}M)")
print(f"Survival:    {100*td/max(1,stats['total']):.1f}%")
print(f"Manual:      {stats['manual_docs']:,} docs, {stats['manual_words']/1e6:.1f}M words")
print(f"Downloaded:  {stats['downloaded_docs']:,} docs, {stats['downloaded_words']/1e6:.1f}M words")
if tw > 0:
    print(f"Manual %:    {100*stats['manual_words']/tw:.1f}%")
print(f"\nDrop reasons:")
for k, v in sorted(stats.items(), key=lambda x: -x[1]):
    if k.startswith("drop_"):
        print(f"  {k[5:]:<18} {v:>10,}  ({100*v/max(1,stats['total']):.1f}%)")
print(f"\nOutput: {OUTPUT_FILE}")
if OUTPUT_FILE.exists():
    print(f"Size: {OUTPUT_FILE.stat().st_size/1e9:.2f} GB")
print("=" * 60)


Input:       17,860,087 docs
Output:      12,499,667 docs, 806,182,413 words (806.2M)
Survival:    70.0%
Manual:      947,475 docs, 97.3M words
Downloaded:  11,552,192 docs, 708.9M words
Manual %:    12.1%

Drop reasons:
  too_short           3,558,463  (19.9%)
  language            1,057,590  (5.9%)
  exact_dup             678,039  (3.8%)
  near_dup               64,457  (0.4%)
  empty                   1,080  (0.0%)
  word_repeat               746  (0.0%)
  menu_like                  45  (0.0%)

Output: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/telugu/datasets/cleaned/telugu_cleaned_corpus.jsonl
Size: 17.96 GB


## 7. Inspect Dropped Docs

Sample 5 dropped docs per filter from the manual sources to validate filter correctness.

In [7]:
import random
random.seed(42)

sample_file = manual_files[0]
drop_buckets = {}

with open(sample_file) as f:
    for i, line in enumerate(f):
        if i >= 5000:
            break
        try:
            text = json.loads(line.strip()).get("text", "")
        except Exception:
            continue
        cleaned = preprocess_manual(text)
        if not cleaned:
            drop_buckets.setdefault("empty", []).append(text[:300])
            continue
        if not passes_language(cleaned):
            drop_buckets.setdefault("language", []).append(cleaned[:300])
            continue
        ok, r = passes_heuristics(cleaned)
        if not ok:
            drop_buckets.setdefault(r, []).append(cleaned[:300])

for reason, samples in drop_buckets.items():
    print(f"\n{'='*55}")
    print(f"Filter: {reason} — {len(samples)} dropped in first 5000 docs of {sample_file.name}")
    print('='*55)
    for j, s in enumerate(random.sample(samples, min(5, len(samples))), 1):
        print(f"  [{j}] {s[:200]}")
        print("  " + "-"*40)



Filter: language — 2 dropped in first 5000 docs of bharatiya_chitrakala_ocr.jsonl
  [1] వ3కాడ౦0 1౬
[షవూగవ౫ఐ 6 (1125ల
గెటివాత[)౫ఎ 6౧౫౧1 ట

|

00-0. ౮౭00400426 142 0016060౧. 091266 0 66209011
  ----------------------------------------
  [2] 5151 ౮ఉం4ఉ౦060౮గ0౮ ౪191144540 [0/4 ౮[1401404410101
(.18గఉగొ/)
ఆడి[141/140//4010441 04, 4౧4149!

న...

న|9౭56 ౧61000 419 ౪౦14౧౧6 0౧ 0 061068 1౧6 ర(6 125 5.200066
0౪66౮06 ౪౦1౪౮౧౧౭ ౪/1 06 0021366 1/- ౧6౧
  ----------------------------------------


## 8. Train / Val / Test Split (90/5/5)

In [8]:
import random
random.seed(42)

train_f = CLEANED_DIR / "telugu_train.jsonl"
val_f   = CLEANED_DIR / "telugu_val.jsonl"
test_f  = CLEANED_DIR / "telugu_test.jsonl"
c = {"train": [0, 0], "val": [0, 0], "test": [0, 0]}

with open(OUTPUT_FILE) as fin, open(train_f,"w") as ft, open(val_f,"w") as fv, open(test_f,"w") as fte:
    for line in fin:
        r = random.random()
        try:
            w = len(json.loads(line).get("text","").split())
        except Exception:
            w = 0
        if   r < 0.90: ft.write(line);  c["train"][0] += 1; c["train"][1] += w
        elif r < 0.95: fv.write(line);  c["val"][0]   += 1; c["val"][1]   += w
        else:          fte.write(line); c["test"][0]  += 1; c["test"][1]  += w

for split, (d, w) in c.items():
    print(f"{split:>5}: {d:>10,} docs  {w:>12,} words  ({w/1e6:.1f}M)")


train: 11,250,137 docs   725,536,917 words  (725.5M)
  val:    623,900 docs    40,384,037 words  (40.4M)
 test:    625,630 docs    40,261,459 words  (40.3M)
